# 02 · lumen against Geant4 and WCSim

Is lumen's light the light Geant4 makes? We check it two ways on Super-K, then on WCTE:

1. **The same event.** Notebook 00's muon and electron: Geant4 tracked their light itself and recorded every photon
   a PMT absorbed (a *hit*). We send the same Geant4 steps through lumen and compare, PMT by PMT, the number of
   hits and when they arrive.
2. **The family's charge check.** Plain Geant4 10.7.4 (`g4ref`) and WCSim (the experiments' Geant4 application for
   water Cherenkov detectors) fired flashes of light of one colour from a few points; `lumen.compare.charge` fires
   the same flashes in lumen and compares every PMT. These reference runs are listed, with their provenance, in
   lumen-wc's manifest (`lumen_wc.REFERENCES`).

Reading the numbers: a **ratio** of totals lumen / reference with its error and **z** = (ratio - 1) / error
(|z| <= 3 passes); a **chi2/ndf** over PMTs, about 1 when the PMTs differ only by chance.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown

import lumen
import lumen_wc
from lumen.compare import charge, event, references, report
from lumen.tools import plot

WORK = Path(os.environ.get("LUMEN_NOTEBOOKS", "lumen-notebooks"))
WORK.mkdir(parents=True, exist_ok=True)


def detector(name):
    """Load a lumen-wc detector, converting it the first time (notebook 01)."""
    path = WORK / f"{name}.lumen"
    if not path.exists():
        lumen.convert(f"lumen_wc:{name}", out=path, data=references.store())
    return path, lumen.load(path)


sk_path, sk = detector("sk")
events = references.product("g4ref-events-wc-933e631d", extra=[lumen_wc.REFERENCES])

## 1. The same event, through lumen and through Geant4

`event.compare` runs the steps through lumen and returns, per PMT, lumen's expected hits and Geant4's hits, and the
arrival times of every hit. lumen counts what Geant4 counts: a photon absorbed at the PMT's photocathode (before
the quantum efficiency). One event has tens of thousands of hits, so each PMT's count fluctuates by its square root;
the pulls should look like a normal distribution of width 1.

In [ ]:
results = {}
for name in ("sk_mu600", "sk_e600"):
    ev = event.read(events / f"{name}.npz")
    res = event.compare(sk, ev, samples=2**18, seed=0)
    results[name] = (ev, res)
    s = res["summary"]
    print(f"{name}: lumen / Geant4 = {s['ratio']:.4f} ± {s['err']:.4f} (z {s['z']:+.1f}); "
          f"chi2/ndf {s['chi2'] / s['ndf']:.2f} over {s['ndf']} PMTs; mean time lumen - Geant4 {s['dt']:+.2f} ns")
    plot.compare(res, title=name)
plt.show()

The muon side by side: Geant4's hits (one event, so grainy) and lumen's expectation.

In [ ]:
ev, res = results["sk_mu600"]
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
top = max(res["geant4"].max(), res["lumen"].max())
plot.display(sk, res["geant4"], a, title="Geant4 hits (one event)", label="hits", log=True, vmax=top)
plot.display(sk, res["lumen"], b, title="lumen (expected hits)", label="hits", log=True, vmax=top)
plt.show()

## 2. The family's charge check: Super-K against Geant4 and WCSim

Five flashes in Super-K, from two references kept in the reference store. Where they differ lumen follows plain
Geant4, so the Geant4 entry (`sk_g4`) passes or fails and WCSim's (`sk`) is reported beside it as a cross-check.
Geant4 ran lumen's **overlap-free export** of the detector (`lumen.compare.ownership`): where the GDML's volumes
overlap, Geant4's choice of owner is undefined, so it runs a GDML in which every shared region has lumen's owner.
`photons_factor` sets lumen's photons as a share of the reference's; we use a tenth to keep this quick, so lumen's
own noise is part of the errors.

In [ ]:
rows = []
for name in ("sk_g4", "sk"):
    spec = references.spec("charge", name, [lumen_wc.REFERENCES])
    gated = references.gated("charge", spec, [lumen_wc.REFERENCES])
    print(f"{name}: reference {spec['reference']} ({'passes or fails' if gated else 'reported'})")
    rows += charge.check(name, detector=sk_path, extra=[lumen_wc.REFERENCES], photons_factor=0.1)
Markdown(report.table(rows))

## WCTE against the same two references

The Water Cherenkov Test Experiment has the same two: WCSim (the experiment's software, on its own Geant4 build)
and plain Geant4 10.7.4 (`g4ref`) on lumen's overlap-free export of the detector. Again the plain Geant4 entry
(`wcte_g4`) passes or fails and WCSim's is reported beside it.

In [ ]:
wcte_path, wcte = detector("wcte")
rows = []
for name in ("wcte", "wcte_g4"):
    spec = references.spec("charge", name, [lumen_wc.REFERENCES])
    gated = references.gated("charge", spec, [lumen_wc.REFERENCES])
    print(f"{name}: reference {spec['reference']} ({'passes or fails' if gated else 'reported'})")
    rows += charge.check(name, detector=wcte_path, extra=[lumen_wc.REFERENCES], photons_factor=0.1)
Markdown(report.table([r for r in rows if r["quantity"] in ("ratio", "truncated weight")]))

**Next:** 03 explains derivatives of a simulation: what they are and how to trust them.